# Introduction

On this notebook I'll try to share some things I've learned here on Kaggle, keeping it as simple as possible. Much of my work here is inspired on @cdeotte's notebooks, you can see that very clear.

Another great inspiration for this was [Alexander's notebook](https://www.kaggle.com/ateplyuk/tpu-tensor-processing-unit-mnist-efficientnet), but I wanted to add few more features as:

* Cross validation (K-Fold)
* Data augmentation
* Some kind of ensemble using folds, as Chris does on his public notebooks
* Keras callbacks (EarlyStopping, ModelCheckpoint, ReduceLROnPlateu)
* Training with class weights
* Using EfficientNet models
* "Noisy student" weights

Few of these features are still missing (you can see that on the TODO list below), but I intend to implement it ASAP.

### TODO:
* Add class_weight

# Notebook configuration

On this section you'll fill the desired configuration as you wish. These are the attributes available:

* DEVICE: "TPU" or "GPU". If you leave it as "TPU", it will check for its availability and, if not available, will switch to GPU. If GPU is not available either, will work with CPU. Please enable your desired Accelerator on your Kaggle notebook settings.
* SEED: the random seed for K-Fold (can be any integer)
* FOLDS: number of folds (K)
* BATCH_SIZE: number of training samples used in one iteration
* EPOCHS: max number of epochs the model will train
* MONITOR: which variable to monitor for setting up callbacks
* MONITOR_MODE: "min" will trigger when MONITOR is lower than previous. "max" is the opposite. "auto" will choose at runtime
* ES_PATIENCE: number of epochs the EarlyStopping will wait for MONITOR_MODE to trigger. If it doesn't trigger, training will stop
* LR_PATIENCE: number of epochs the ReduceLROnPlateu will wait for MONITOR_MODE to trigger. If it doesn't trigger, the learning rate will be reduced by LR_FACTOR
* LR_FACTOR: constant for multiplication of learning rate
* EFF_NET: which EfficientNet you'd like to use. Choose from 0 to 8. 0 is EfficientNetB0, 1 is EfficientNetB1, and so on. 8 is EfficientNetL2
* EFF_NET_WEIGHTS: which weights to use for fine tuning. "imagenet" and "noisy-student" are available
* LABEL_SMOOTHING: regularization parameter. 0 is default
* VERBOSE: verbose of printing: 0 for no printing, 1 for debugging, 2 for commiting

In [1]:
#
# Notebook configuration
#

DEVICE            = "TPU"  # Any value in ["TPU", "GPU"]
SEED              = 8080
FOLDS             = 5
FOLD_WEIGHTS      = [1./FOLDS]*FOLDS
BATCH_SIZE        = 256
EPOCHS            = 5000
MONITOR           = "val_loss"
MONITOR_MODE      = "min"
ES_PATIENCE       = 5
LR_PATIENCE       = 0
LR_FACTOR         = 0.5
EFF_NET           = 3
EFF_NET_WEIGHTS   = 'noisy-student'
LABEL_SMOOTHING   = 0.1
VERBOSE           = 1

# Initial setup

Here we'll install EfficientNet dependencies and setup our enviroment according to DEVICE chosen

In [2]:
!pip install -q efficientnet >> /dev/null

You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.backend as K
import efficientnet.tfkeras as efn
from sklearn.model_selection import KFold
from keras.preprocessing.image import ImageDataGenerator

In [4]:
if DEVICE == "TPU":
    print("connecting to TPU...")
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        print('Running on TPU ', tpu.master())
    except ValueError:
        print("Could not connect to TPU")
        tpu = None

    if tpu:
        try:
            print("initializing  TPU ...")
            tf.config.experimental_connect_to_cluster(tpu)
            tf.tpu.experimental.initialize_tpu_system(tpu)
            strategy = tf.distribute.experimental.TPUStrategy(tpu)
            print("TPU initialized")
        except _:
            print("failed to initialize TPU")
    else:
        DEVICE = "GPU"

if DEVICE != "TPU":
    print("Using default strategy for CPU and single GPU")
    strategy = tf.distribute.get_strategy()

if DEVICE == "GPU":
    print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
    

AUTO     = tf.data.experimental.AUTOTUNE
REPLICAS = strategy.num_replicas_in_sync
print(f'REPLICAS: {REPLICAS}')

connecting to TPU...
Running on TPU  grpc://10.0.0.2:8470
initializing  TPU ...
TPU initialized
REPLICAS: 8


# Loading data

Here we'll load train and test data, converting it to a float32 np array and then few pre-processing:

* Normalize X data;
* Padding the images so they fit EfficientNet minimum size (32x32);
* Triple X data so it becomes a 3-channel (could be RGB, but our image has a single color) image, required by EfficientNet;
* Hot-encode y data so we can use a 10-node softmax output layer on our neural network.

In [5]:
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
train.describe()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,42000.000000,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,42000.0,...,42000.000000,42000.000000,42000.000000,42000.00000,42000.000000,42000.000000,42000.0,42000.0,42000.0,42000.0
mean,4.456643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.219286,0.117095,0.059024,0.02019,0.017238,0.002857,0.0,0.0,0.0,0.0
std,2.887730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.312890,4.633819,3.274488,1.75987,1.894498,0.414264,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,254.000000,254.000000,253.000000,253.00000,254.000000,62.000000,0.0,0.0,0.0,0.0


In [6]:
# Getting X data
X = train.drop(labels=['label'], axis=1)
X = X.astype('float32')

# Data normalization
X = X / 255

# 2D, 1 channel reshape
X = X.values.reshape(X.shape[0],28,28,1)

# Padding image so it fits on EfficientNet minimum size (32x32)
X = np.pad(X, ((0,0), (2,2), (2,2), (0,0)), mode='constant')

# Copying channel so it becomes a 3-channel image (32x32x3), required by EfficientNet
X = np.squeeze(X, axis=-1)
X = stacked_img = np.stack((X,)*3, axis=-1)

# Checking shape, it must be (n_images, 32, 32, 3)
X.shape

(42000, 32, 32, 3)

In [7]:
# Getting labels
y = train['label'].values.astype('float32')

# One-hot encoding labels
y = tf.keras.utils.to_categorical(y, 10)

y

array([[0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [8]:
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
test.describe()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
count,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,28000.0,...,28000.000000,28000.000000,28000.000000,28000.000000,28000.000000,28000.0,28000.0,28000.0,28000.0,28000.0
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.164607,0.073214,0.028036,0.011250,0.006536,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.473293,3.616811,1.813602,1.205211,0.807475,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,253.000000,254.000000,193.000000,187.000000,119.000000,0.0,0.0,0.0,0.0,0.0


In [9]:
# Getting X data
X_test = test.astype('float32')

# Data normalization
X_test = X_test / 255

# 2D, 1 channel reshape
X_test = X_test.values.reshape(X_test.shape[0],28,28,1)

# Padding image so it fits on EfficientNet minimum size (32x32)
X_test = np.pad(X_test, ((0,0), (2,2), (2,2), (0,0)), mode='constant')

# Copying channel so it becomes a 3-channel image (32x32x3), required by EfficientNet
X_test = np.squeeze(X_test, axis=-1)
X_test = stacked_img = np.stack((X_test,)*3, axis=-1)

# Checking shape, it must be (n_images, 32, 32, 3)
X_test.shape

(28000, 32, 32, 3)

# Augmenting data

One smart move for datasets with low amount of data is to augment it. Here, I'll use keras `ImageDataGenerator`, as shown on one of Chris' notebooks on this competition and [here](https://www.kaggle.com/yassineghouzam/introduction-to-cnn-keras-0-997-top-6)

In [10]:
# Got this from https://www.kaggle.com/yassineghouzam/introduction-to-cnn-keras-0-997-top-6
datagen = ImageDataGenerator(
    featurewise_center=False,              # set input mean to 0 over the dataset
    samplewise_center=False,               # set each sample mean to 0
    featurewise_std_normalization=False,   # divide inputs by std of the dataset
    samplewise_std_normalization=False,    # divide each input by its std
    zca_whitening=False,                   # apply ZCA whitening
    rotation_range=10,                     # randomly rotate images in the range (degrees, 0 to 180)
    zoom_range = 0.1,                      # Randomly zoom image 
    width_shift_range=0.1,                 # randomly shift images horizontally (fraction of total width)
    height_shift_range=0.1,                # randomly shift images vertically (fraction of total height)
    horizontal_flip=False,                 # randomly flip images
    vertical_flip=False                    # randomly flip images
)

# Building model

Here is where you can edit the model to your needs. For this notebook, I'll just use a simple head above the EfficientNet model. In my (very short) experience, adding too much above a pre-trained model tends to underfit the last layers and then get worse results.

In [11]:
eff_nets = [
    efn.EfficientNetB0,
    efn.EfficientNetB1,
    efn.EfficientNetB2,
    efn.EfficientNetB3,
    efn.EfficientNetB4,
    efn.EfficientNetB5,
    efn.EfficientNetB6,
    efn.EfficientNetB7,
    efn.EfficientNetL2,
]

def build_model ():
    inp = tf.keras.layers.Input(shape=(X.shape[1], X.shape[2], X.shape[3]))
    oup = eff_nets[EFF_NET](
        input_shape=(X.shape[1], X.shape[2], X.shape[3]),
        weights=EFF_NET_WEIGHTS,
        include_top=False,
    )(inp)
    oup = tf.keras.layers.GlobalAveragePooling2D()(oup)
    oup = tf.keras.layers.Dense(512, activation='linear')(oup)
    oup = tf.keras.layers.Activation('relu')(oup)
    oup = tf.keras.layers.Dropout(0.5)(oup)
    oup = tf.keras.layers.Dense(10, activation='linear')(oup)
    oup = tf.keras.layers.Activation('softmax')(oup)
    
    model = tf.keras.Model (inputs=[inp], outputs=[oup])
    
    loss = tf.keras.losses.CategoricalCrossentropy(
        from_logits=False,
        label_smoothing=LABEL_SMOOTHING,
    )
    
    opt = tf.keras.optimizers.Nadam(learning_rate=3e-4)
    
    model.compile(optimizer=opt,loss=loss,metrics=['acc'])
    
    return model

build_model().summary()

43933696/43933088 [==============================] - 1s 0us/step
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 32, 32, 3)]       0         
_________________________________________________________________
efficientnet-b3 (Model)      (None, 1, 1, 1536)        10783528  
_________________________________________________________________
global_average_pooling2d (Gl (None, 1536)              0         
_________________________________________________________________
dense (Dense)                (None, 512)               786944    
_________________________________________________________________
activation (Activation)      (None, 512)               0         
_________________________________________________________________
dropout (Dropout)            (None, 512)               0         
______________________________________________________________

# Training

This is the training part. Here, for each fold, we'll train the model until EarlyStopping tells us to stop or EPOCHS is reached. Then, we'll predict on the validation set, compute its accuracy (sanity check) and then predict the full test set, multiplying each fold with its weights.

PS: There's a workaround I've implemented because of [this issue](https://github.com/tensorflow/tensorflow/issues/34346) with TPUs.

In [12]:
%%time

# USE VERBOSE=0 for silent, VERBOSE=1 for interactive, VERBOSE=2 for commit

oof   = np.zeros((X.shape[0], y.shape[1]))
preds = np.zeros((X_test.shape[0], y.shape[1]))

skf = KFold(n_splits=FOLDS,shuffle=True,random_state=SEED)

for fold,(idxT,idxV) in enumerate(skf.split(X)):

    if DEVICE=='TPU':
        if tpu: tf.tpu.experimental.initialize_tpu_system(tpu)
            
    print('#'*25)
    print('### FOLD %i'%(fold+1))
    print('#'*25)
    
    K.clear_session()
    with strategy.scope():
        model = build_model()
        
    weights_filename='fold-%i.h5'%fold
        
    # Save best model for each fold
    sv = tf.keras.callbacks.ModelCheckpoint(
        weights_filename, monitor=MONITOR, verbose=VERBOSE, save_best_only=True,
        save_weights_only=True, mode=MONITOR_MODE, save_freq='epoch')

    # Learning rate reduction
    lrr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR,
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        verbose=VERBOSE,
        mode=MONITOR_MODE
    )
    
    # Early stopping
    es = tf.keras.callbacks.EarlyStopping(
        monitor=MONITOR,
        patience=ES_PATIENCE,
        verbose=VERBOSE,
        mode=MONITOR_MODE,
    )
    
    # Datagen workaround
    print ('Generating train data...')
    i = 0
    datagen.fit(X[idxT])
    steps = 2 * (X[idxT].shape[0] // BATCH_SIZE)
    X_train = None
    y_train = None
    with tqdm(total=steps) as pbar:
        for arr in datagen.flow(X[idxT], y[idxT], batch_size=BATCH_SIZE):
            if X_train is None:
                X_train = arr[0]
                y_train = arr[1]
            else:
                X_train = np.concatenate((X_train, arr[0]))
                y_train = np.concatenate((y_train, arr[1]))
            i += 1
            pbar.update(1)
            if i >= steps:
                break
                
#     # Class weights
#     w = 1 / np.sum(y_train, axis=0)
#     w *= 1 / np.min(w)
#     cw = {
#         np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0]) : w[0]
#     }
    
    # Train
    print('Training...')
    history = model.fit(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks = [
            sv,
            lrr,
            es,
        ],
        validation_data=(
            X[idxV], y[idxV]
        ),
        verbose=VERBOSE,
#         class_weight = { 0: 1, 1: 56 }
    )

    print('Loading best model...')
    model.load_weights('fold-%i.h5'%fold)
    
    print('Predicting OOF...')
    oof[idxV,] = model.predict([X[idxV]],verbose=VERBOSE)
    
    print('Predicting Test...')
    preds += (model.predict([X_test], verbose=VERBOSE) * FOLD_WEIGHTS[fold])

    acc_sum = 0
    for k in idxV:
        if np.argmax(oof[k]) == np.argmax(y[k]):
            acc_sum += 1
    print('>>>> FOLD {} Accuracy = {:.4f}%'.format(fold+1,acc_sum/len(idxV)*100))
    print()

#########################
### FOLD 1
#########################
Generating train data...


100%|██████████| 262/262 [01:19<00:00,  3.30it/s]


Training...
Epoch 1/5000
262/262 [==============================] - ETA: 0s - acc: 0.6489 - loss: 1.3445
Epoch 00001: val_loss improved from inf to 1.28309, saving model to fold-0.h5
262/262 [==============================] - 56s 215ms/step - acc: 0.6489 - loss: 1.3445 - val_acc: 0.6581 - val_loss: 1.2831 - lr: 3.0000e-04
Epoch 2/5000
262/262 [==============================] - ETA: 0s - acc: 0.9147 - loss: 0.7426
Epoch 00002: val_loss improved from 1.28309 to 0.63930, saving model to fold-0.h5
262/262 [==============================] - 20s 74ms/step - acc: 0.9147 - loss: 0.7426 - val_acc: 0.9498 - val_loss: 0.6393 - lr: 3.0000e-04
Epoch 3/5000
262/262 [==============================] - ETA: 0s - acc: 0.9503 - loss: 0.6505
Epoch 00003: val_loss improved from 0.63930 to 0.56048, saving model to fold-0.h5
262/262 [==============================] - 20s 76ms/step - acc: 0.9503 - loss: 0.6505 - val_acc: 0.9810 - val_loss: 0.5605 - lr: 3.0000e-04
Epoch 4/5000
262/262 [========================

100%|██████████| 262/262 [01:19<00:00,  3.28it/s]


Training...
Epoch 1/5000
262/262 [==============================] - ETA: 0s - acc: 0.6648 - loss: 1.3133
Epoch 00001: val_loss improved from inf to 1.43891, saving model to fold-1.h5
262/262 [==============================] - 56s 213ms/step - acc: 0.6648 - loss: 1.3133 - val_acc: 0.6249 - val_loss: 1.4389 - lr: 3.0000e-04
Epoch 2/5000
262/262 [==============================] - ETA: 0s - acc: 0.9231 - loss: 0.7247
Epoch 00002: val_loss improved from 1.43891 to 0.69464, saving model to fold-1.h5
262/262 [==============================] - 20s 76ms/step - acc: 0.9231 - loss: 0.7247 - val_acc: 0.9264 - val_loss: 0.6946 - lr: 3.0000e-04
Epoch 3/5000
262/262 [==============================] - ETA: 0s - acc: 0.9539 - loss: 0.6411
Epoch 00003: val_loss improved from 0.69464 to 0.57108, saving model to fold-1.h5
262/262 [==============================] - 21s 80ms/step - acc: 0.9539 - loss: 0.6411 - val_acc: 0.9755 - val_loss: 0.5711 - lr: 3.0000e-04
Epoch 4/5000
262/262 [========================

...

Training...
Epoch 1/5000
262/262 [==============================] - ETA: 0s - acc: 0.6561 - loss: 1.3331
Epoch 00001: val_loss improved from inf to 1.56086, saving model to fold-2.h5
262/262 [==============================] - 56s 215ms/step - acc: 0.6561 - loss: 1.3331 - val_acc: 0.5195 - val_loss: 1.5609 - lr: 3.0000e-04
Epoch 2/5000
262/262 [==============================] - ETA: 0s - acc: 0.9110 - loss: 0.7544
Epoch 00002: val_loss improved from 1.56086 to 0.68442, saving model to fold-2.h5
262/262 [==============================] - 21s 78ms/step - acc: 0.9110 - loss: 0.7544 - val_acc: 0.9360 - val_loss: 0.6844 - lr: 3.0000e-04
Epoch 3/5000
262/262 [==============================] - ETA: 0s - acc: 0.9482 - loss: 0.6557
Epoch 00003: val_loss improved from 0.68442 to 0.56236, saving model to fold-2.h5
262/262 [==============================] - 20s 76ms/step - acc: 0.9482 - loss: 0.6557 - val_acc: 0.9793 - val_loss: 0.5624 - lr: 3.0000e-04
Epoch 4/5000
262/262 [========================

100%|██████████| 262/262 [01:20<00:00,  3.27it/s]


Training...
Epoch 1/5000
262/262 [==============================] - ETA: 0s - acc: 0.6763 - loss: 1.2878
Epoch 00001: val_loss improved from inf to 1.73867, saving model to fold-3.h5
262/262 [==============================] - 58s 221ms/step - acc: 0.6763 - loss: 1.2878 - val_acc: 0.3663 - val_loss: 1.7387 - lr: 3.0000e-04
Epoch 2/5000
262/262 [==============================] - ETA: 0s - acc: 0.9263 - loss: 0.7172
Epoch 00002: val_loss improved from 1.73867 to 0.64594, saving model to fold-3.h5
262/262 [==============================] - 22s 82ms/step - acc: 0.9263 - loss: 0.7172 - val_acc: 0.9495 - val_loss: 0.6459 - lr: 3.0000e-04
Epoch 3/5000
262/262 [==============================] - ETA: 0s - acc: 0.9568 - loss: 0.6347
Epoch 00003: val_loss improved from 0.64594 to 0.56826, saving model to fold-3.h5
262/262 [==============================] - 22s 82ms/step - acc: 0.9568 - loss: 0.6347 - val_acc: 0.9765 - val_loss: 0.5683 - lr: 3.0000e-04
Epoch 4/5000
262/262 [========================

100%|██████████| 262/262 [01:21<00:00,  3.22it/s]


Training...
Epoch 1/5000
262/262 [==============================] - ETA: 0s - acc: 0.6603 - loss: 1.3253
Epoch 00001: val_loss improved from inf to 1.71421, saving model to fold-4.h5
262/262 [==============================] - 58s 219ms/step - acc: 0.6603 - loss: 1.3253 - val_acc: 0.4273 - val_loss: 1.7142 - lr: 3.0000e-04
Epoch 2/5000
262/262 [==============================] - ETA: 0s - acc: 0.9180 - loss: 0.7379
Epoch 00002: val_loss improved from 1.71421 to 0.65983, saving model to fold-4.h5
262/262 [==============================] - 21s 81ms/step - acc: 0.9180 - loss: 0.7379 - val_acc: 0.9470 - val_loss: 0.6598 - lr: 3.0000e-04
Epoch 3/5000
262/262 [==============================] - ETA: 0s - acc: 0.9524 - loss: 0.6455
Epoch 00003: val_loss improved from 0.65983 to 0.57376, saving model to fold-4.h5
262/262 [==============================] - 22s 83ms/step - acc: 0.9524 - loss: 0.6455 - val_acc: 0.9757 - val_loss: 0.5738 - lr: 3.0000e-04
Epoch 4/5000
262/262 [========================

# Submitting predictions

On this section, we'll just use `argmax` for getting the predictions from an array `[0.1 0.01 0.3 ... 0.85]` to an integer `0 .. 9` and then generate the CSV file we'll submit.

In [13]:
# Getting predictions output on the demanded format
final_predictions = pd.Series(np.argmax(preds, axis=1), name="Label")

# Generating dataframe on the demanded format
submission = pd.concat([pd.Series(range(1,28001),name = "ImageId"),final_predictions], axis=1)

# Saving file
submission.to_csv("submission.csv",index=False)

# Printing few samples
submission.head()

,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3
